# QLoRA fine-tune of Qwen2.5-3B-Instruct for EV (outline, not run)

**Status: an outline. No cell in this notebook has been run, and nothing here has been measured.** It is meant to be run on
Colab or Kaggle (a free T4 has 15 GB; the owner's laptop has 4 GB of VRAM and cannot train a 3B model).

**Why bother.** `qwen2.5:3b` follows the persona's tone rules loosely (D29: it asks a question in about 7 of 10 replies whatever
the wording) and its tool calling shifts with persona edits (D22). Fine-tuning could fix both. It could also make things worse, so
the notebook ends with a gate: the new model replaces the old one only if it beats it on EV's own evals.

**What to train on.** `scripts/finetune/export_data.py` (redacted, opt-in, never turns that quoted outside text). Today that is the
72 hand-written intent examples, which is far too little by itself: plan on at least a few thousand varied turns, which means
either collecting your own (a separate opt-in, not built) or generating synthetic ones, and *reviewing them*. Synthetic data from
another model teaches this model that model's habits.

**What not to train on.** Anything from the web, the screen, files or notes (the export drops those turns), and the safety
behaviour: the gate, not the model, decides what is allowed, and a fine-tune must not be trusted to refuse anything.

## 1. Environment (Colab / Kaggle)

In [ ]:
!pip -q install "unsloth" "trl>=0.12" "peft" "bitsandbytes" "datasets" "accelerate"
import torch; print(torch.cuda.get_device_name(0), torch.cuda.mem_get_info())

## 2. Data
Upload `train.jsonl` and `val.jsonl` from `data/training/export/` (chat format with `tool_calls`). Check by eye: print twenty
examples, look for names, places, anything private that the redaction missed. **Split by utterance, not by row**, and keep the
intent eval's 72 utterances out of training entirely if they are your gate (the export includes them as seeds; drop them with
`--no-evals` when you want a clean test).

In [ ]:
from datasets import load_dataset
ds = load_dataset("json", data_files={"train": "train.jsonl", "validation": "val.jsonl"})
print(ds)
for row in ds["train"].shuffle(seed=1).select(range(20)):
    print(row["messages"][1]["content"], "->", row["messages"][2].get("tool_calls") or row["messages"][2]["content"])

## 3. Model and LoRA config
4-bit base, rank 16 on all attention and MLP projections, small learning rate, 1 to 3 epochs. More epochs on a few thousand rows
mostly memorises. Starting values below are common defaults, **not tuned**.

In [ ]:
from unsloth import FastLanguageModel
model, tok = FastLanguageModel.from_pretrained("Qwen/Qwen2.5-3B-Instruct", max_seq_length=2048, load_in_4bit=True)
model = FastLanguageModel.get_peft_model(
    model, r=16, lora_alpha=16, lora_dropout=0.0,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])

## 4. Train (loss on the assistant turn only)

In [ ]:
from trl import SFTTrainer, SFTConfig
def to_text(batch):
    return {"text": [tok.apply_chat_template(m, tokenize=False) for m in batch["messages"]]}
train = ds["train"].map(to_text, batched=True)
trainer = SFTTrainer(model=model, tokenizer=tok, train_dataset=train, args=SFTConfig(
    per_device_train_batch_size=4, gradient_accumulation_steps=4, num_train_epochs=2, learning_rate=1e-4,
    lr_scheduler_type="cosine", warmup_ratio=0.05, logging_steps=10, output_dir="out", dataset_text_field="text", seed=7))
trainer.train()

## 5. Quick check on held-out turns
Generate for the validation prompts and compare the tool name and arguments with the expected ones. This is a smoke test, not the
gate: the gate is EV's own evals (section 8), run through Ollama with the real prompt and tool list.

In [ ]:
FastLanguageModel.for_inference(model)
# TODO: loop over ds["validation"], generate, parse the tool call, count exact matches on name and on arguments.

## 6. Merge and export to GGUF
Merge the adapter into the 16-bit base and convert with llama.cpp. `q4_k_m` is the usual size/quality trade for a 3B model, and it
is the quantisation Ollama's own `qwen2.5:3b` uses, so latency should be comparable. **Not measured.**

In [ ]:
model.save_pretrained_gguf("evie-qwen25-3b", tok, quantization_method="q4_k_m")

## 7. Bring it to the laptop
Download the `.gguf`, then on the laptop (conda env `ev`; Ollama on 127.0.0.1):

    # Modelfile
    FROM ./evie-qwen25-3b-q4_k_m.gguf
    PARAMETER temperature 0.6
    PARAMETER num_ctx 6144

    ollama create evie-ft -f Modelfile

The chat template must match what Ollama expects for Qwen2.5 tool calls (check `ollama show evie-ft --template`); a mismatch
breaks tool calling quietly, which the evals below would show.

## 8. The gate: it replaces the base model only if it wins
Run each with the base model and with `evie-ft`, three times each (the spread between runs is several points):

    python -m ev.evals.run intent  --model evie-ft
    python -m ev.evals.run persona --model evie-ft
    python -m ev.evals.run memory  --model evie-ft
    python -m ev.evals.run security            # with the fine-tuned model selected in ev.local.toml

Ship it only if **all** hold: intent no worse than the base (and the model-only `remember_fact` rate no worse), persona pass rate
no worse and the question rate and callback rate better, security **0 bypasses and no new failures**, VRAM and first-reply
latency within 10% of the base. Otherwise keep `qwen2.5:3b`. Keep the base model in Ollama as the fallback (`llm.fallback_model`).

Things that would make this a bad idea: too little data (it learns the wording of a few examples), data that includes your own
tics (it becomes an echo), or a model that scores well on the evals above because the evals' utterances leaked into training.